# 27. 장르별 긍정률 분포 — 첫 반응군(리뷰 10~49개)

**분석 목적:** 첫 반응군(리뷰 10~49개) 안에서, 장르별 유저 만족도(긍정률)의 분포를 중앙값 기준으로 비교한다.
인디 개발사가 장르를 선택할 때 '어떤 장르가 초기 유저 만족도를 확보하기 유리한가'에 대한 데이터 기반 참고점을 제공한다.

**사용 데이터:** `data/preprocessed/steam_indie_games_graded.csv` (리뷰 10~49개 필터, 2023~2025년, EA·F2P 제외)

**분석 흐름:**
1. 라이브러리 로드 및 공통 설정
2. 데이터 로드 — 첫 반응군 필터링 및 장르 explode
3. 장르별 긍정률 중앙값 바 차트 (슬라이드 27 메인 시각화)
4. 장르별 긍정률 박스플롯 (분포 상세)
5. 장르별 긍정률 통계 요약

---

> **중앙값을 사용하는 이유**
>
> 첫 반응군의 장르별 긍정률 분포는 좌편향(negative skew)이 강하다 — 대부분의 게임이 긍정률 상단에 몰려 있고,
> 소수의 저긍정률 게임이 평균을 아래로 잡아당긴다.
> 평균은 이러한 이상치의 영향을 받아 실제 전형적인 게임 수준보다 낮게 나타나므로,
> 중앙값이 '이 장르의 전형적인 긍정률'을 더 정확하게 나타낸다.
>
> | 장르 | 평균 | 중앙값 | 차이 |
> |------|------|--------|------|
> | Action | 84.5% | 90.0% | −5.5%p |
> | Casual | 85.5% | 90.9% | −5.4%p |
> | Simulation | 78.1% | 82.4% | −4.3%p |

## 1. 라이브러리 로드 및 공통 설정

In [1]:
import ast
import warnings

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']

PALETTE = [
    '#4C72B0', '#DD8452', '#55A868', '#C44E52',
    '#8172B2', '#937860', '#DA8BC3', '#8C8C8C'
]
COLOR_MAP = {g: PALETTE[i] for i, g in enumerate(TARGET_GENRES)}

## 2. 데이터 로드 — 첫 반응군 필터링 및 장르 explode

In [2]:
games = pd.read_csv('../../../data/preprocessed/steam_indie_games_graded.csv')

if 'positive_rate' not in games.columns:
    games['positive_rate'] = games['positive'] / games['total_reviews'] * 100

# 첫 반응군: 리뷰 10~49개
first_response = games[(games['total_reviews'] >= 10) & (games['total_reviews'] <= 49)].copy()

first_response['genres_list'] = first_response['genres'].apply(
    lambda g: ast.literal_eval(g) if pd.notna(g) else []
)
first_response['genres_filtered'] = first_response['genres_list'].apply(
    lambda gl: [g for g in gl if g in TARGET_GENRES]
)

with_genre = first_response[first_response['genres_filtered'].map(len) > 0].copy()
df = (
    with_genre
    .explode('genres_filtered')
    .rename(columns={'genres_filtered': 'genre'})
    .reset_index(drop=True)
)

print(f'첫 반응군 전체    : {len(first_response):,}개')
print(f'장르 있는 게임    : {len(with_genre):,}개')
print(f'explode 후 행 수  : {len(df):,}행 (중복 포함)')
print()
print('장르별 게임 수:')
print(df['genre'].value_counts().to_string())

첫 반응군 전체    : 4,840개
장르 있는 게임    : 4,840개
explode 후 행 수  : 10,221행 (중복 포함)

장르별 게임 수:
genre
Adventure     2441
Casual        2304
Action        2197
Simulation    1100
Strategy       938
RPG            874
Sports         189
Racing         178


## 3. 장르별 긍정률 중앙값 바 차트 (슬라이드 27 메인 시각화)

In [3]:
genre_median = (
    df.groupby('genre')['positive_rate']
    .median()
    .sort_values(ascending=False)
    .reset_index()
)
genre_median.columns = ['genre', 'median_positive_rate']

fig = px.bar(
    genre_median,
    x='genre', y='median_positive_rate',
    color='genre',
    color_discrete_map=COLOR_MAP,
    text='median_positive_rate',
    title='장르별 긍정률 중앙값 — 첫 반응군 (리뷰 10~49개)<br><sub>다중 장르 중복 집계 / 중앙값 기준 내림차순 정렬</sub>',
    labels={'genre': '장르', 'median_positive_rate': '긍정률 중앙값 (%)'},
    category_orders={'genre': genre_median['genre'].tolist()},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.add_hline(
    y=80, line_dash='dash', line_color='red',
    annotation_text='80% (Very Positive 기준)', annotation_position='top right'
)
fig.update_layout(
    showlegend=False,
    yaxis=dict(range=[0, 105], title='긍정률 중앙값 (%)'),
    height=480,
)
fig.show()

**해석:** 첫 반응군(리뷰 10~49개) 안에서 모든 장르의 긍정률 중앙값은 80% 이상으로, 첫 반응을 확보한 게임들은 장르에 관계없이 기본적인 유저 만족도 수준을 달성하고 있다. Casual이 중앙값 기준 가장 높고, Simulation이 가장 낮다. 즉, 첫 반응군 내 만족도 차이는 절대적 수준보다 장르 특성(유저 기대치, 완성도 기준)에서 비롯된다.

## 4. 장르별 긍정률 박스플롯 (분포 상세)

In [4]:
genre_order = genre_median['genre'].tolist()  # 중앙값 내림차순 순서 유지

fig = px.box(
    df,
    x='genre', y='positive_rate',
    category_orders={'genre': genre_order},
    color='genre',
    color_discrete_map=COLOR_MAP,
    points=False,
    title='장르별 긍정률 분포 — 첫 반응군 (리뷰 10~49개)<br><sub>다중 장르 중복 집계 / 중앙값 내림차순 정렬</sub>',
    labels={'genre': '장르', 'positive_rate': '긍정률 (%)'},
)
fig.add_hline(
    y=80, line_dash='dash', line_color='red',
    annotation_text='80% (Very Positive 기준)', annotation_position='top right'
)
fig.update_layout(
    showlegend=False,
    yaxis=dict(range=[0, 110], title='긍정률 (%)'),
    height=480,
)
fig.show()

**해석:** 박스플롯에서 모든 장르의 IQR(25~75%) 상단이 100%에 가깝고, 하단이 70~80% 수준이다. Simulation은 IQR 범위가 가장 넓어 품질 편차가 크다. 좌편향(lower tail) 이상치가 모든 장르에 존재하나 수가 적어, 평균보다 중앙값이 해당 장르의 전형적 긍정률을 더 정확하게 나타낸다.

## 5. 장르별 긍정률 통계 요약

In [5]:
stats = (
    df.groupby('genre')['positive_rate']
    .agg(
        게임수='count',
        중앙값='median',
        평균='mean',
        표준편차='std',
        Q25=lambda x: x.quantile(0.25),
        Q75=lambda x: x.quantile(0.75),
        왜도='skew',
    )
    .round(2)
)
stats['평균_중앙값_차이'] = (stats['평균'] - stats['중앙값']).round(2)
stats = stats.sort_values('중앙값', ascending=False)

print('장르별 긍정률 통계 (첫 반응군, 중앙값 내림차순):')
display(stats)

장르별 긍정률 통계 (첫 반응군, 중앙값 내림차순):


,게임수,중앙값,평균,표준편차,Q25,Q75,왜도,평균_중앙값_차이
genre,,,,,,,,
Casual,2304,90.91,85.52,16.69,78.95,100.00,-1.62,-5.39
Action,2197,90.00,84.51,17.03,76.92,100.00,-1.51,-5.49
Strategy,938,90.00,84.36,16.22,76.52,96.43,-1.40,-5.64
Adventure,2441,88.89,83.54,16.75,75.00,96.30,-1.36,-5.35
RPG,874,88.52,82.72,17.48,73.33,95.83,-1.29,-5.80
Racing,178,85.71,81.85,17.56,72.22,96.99,-0.98,-3.86
Sports,189,84.62,81.69,16.75,72.73,95.45,-1.17,-2.93
Simulation,1100,82.35,78.09,19.15,67.86,92.86,-1.04,-4.26


**해석:**
- **왜도(skew):** 모든 장르에서 음수(−0.98 ~ −1.62)로 좌편향이 확인된다 — 다수의 게임이 고긍정률 구간에 집중하고 소수의 저긍정률 게임이 분포 왼쪽 꼬리를 형성한다.
- **평균_중앙값_차이:** 모든 장르에서 평균이 중앙값보다 낮다(−3 ~ −6%p) — 저긍정률 이상치가 평균을 끌어내린 결과다. 이 차이가 중앙값을 대표 지표로 선택한 근거다.
- **표준편차:** Simulation(19.2)이 가장 높고 Strategy(16.2)가 가장 낮다 — Simulation은 초기 반응군 안에서도 게임 품질 편차가 크다는 것을 의미한다.